In [12]:
# Cài đặt ultralytics nếu chưa có (môi trường Kaggle/Colab)
!pip install ultralytics
import ultralytics
ultralytics.checks() 

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6582.4/8062.4 GB disk)


In [13]:
import os
import cv2
from tqdm.notebook import tqdm

# Cấu hình đường dẫn gốc
ROOT_DIR = '/kaggle/input/face-detection-dataset'
TRAIN_IMG_DIR = os.path.join(ROOT_DIR, 'images/train')
VAL_IMG_DIR = os.path.join(ROOT_DIR, 'images/val')
TRAIN_LBL_DIR = os.path.join(ROOT_DIR, 'labels/train')
VAL_LBL_DIR = os.path.join(ROOT_DIR, 'labels/val')

def create_verified_list(img_dir, lbl_dir, out_filename):
    print(f"--- Đang kiểm tra sâu: {img_dir} ---")
    valid_paths = []
    error_count = 0
    
    files = sorted(os.listdir(img_dir))
    
    for fname in tqdm(files):
        if fname.lower().endswith(('.jpg', '.png', '.jpeg')):
            fpath = os.path.join(img_dir, fname)
            
            # 1. Kiểm tra Label tồn tại không
            stem = os.path.splitext(fname)[0]
            lpath = os.path.join(lbl_dir, stem + '.txt')
            if not os.path.exists(lpath):
                continue
            
            # 1.5 CHECK NỘI DUNG LABEL (CỰC QUAN TRỌNG)
            try:
                with open(lpath, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) != 5:
                            raise ValueError("Label sai format")
            
                        cls, x, y, w, h = map(float, parts)
            
                        if cls < 0 or cls >= 1:   # nc = 1
                            raise ValueError("Class id sai")
            
                        if not (0 <= x <= 1 and 0 <= y <= 1):
                            raise ValueError("Tọa độ center out of range")
            
                        if not (0 < w <= 1 and 0 < h <= 1):
                            raise ValueError("BBox size out of range")
            
            except Exception as e:
                print(f"Label lỗi ({fname}): {e}")
                error_count += 1
                continue
            
            # 2. CHECK QUAN TRỌNG: Thử decode ảnh
            try:
                img = cv2.imread(fpath)
                if img is None:
                    print(f"Lỗi ảnh hỏng: {fname}")
                    error_count += 1
                    continue
            
                valid_paths.append(fpath)
            
            except Exception as e:
                print(f"Crash khi đọc ảnh {fname}: {e}")
                error_count += 1

    # Ghi file list
    out_path = os.path.join('/kaggle/working', out_filename)
    with open(out_path, 'w') as f:
        f.write('\n'.join(valid_paths))
        
    print(f"-> Đã lọc xong {out_filename}. Số ảnh lỗi/hỏng: {error_count}")

# Chạy tạo list
create_verified_list(TRAIN_IMG_DIR, TRAIN_LBL_DIR, 'train_verified.txt')
create_verified_list(VAL_IMG_DIR, VAL_LBL_DIR, 'val_verified.txt')

import random

TRAIN_LIST = '/kaggle/working/train_verified.txt'
EVAL_LIST  = '/kaggle/working/eval_300.txt'

# Đọc danh sách train đã verify
with open(TRAIN_LIST, 'r') as f:
    train_paths = f.read().splitlines()

assert len(train_paths) >= 300, "Train set không đủ 300 ảnh"

# Lấy ngẫu nhiên 300 ảnh
eval_paths = random.sample(train_paths, 300)

# Ghi ra file eval
with open(EVAL_LIST, 'w') as f:
    f.write('\n'.join(eval_paths))

print(f"✅ Đã tạo eval_300.txt với {len(eval_paths)} ảnh")

# Tạo file yaml mới
import yaml
yaml_data = {
    'path': '/kaggle/working',
    'train': 'train_verified.txt',
    'val': 'val_verified.txt',
    'test': 'eval_300.txt',   
    'nc': 1,
    'names': ['face']
}
with open('/kaggle/working/data_final.yaml', 'w') as f:
    yaml.dump(yaml_data, f)

--- Đang kiểm tra sâu: /kaggle/input/face-detection-dataset/images/train ---


  0%|          | 0/13386 [00:00<?, ?it/s]

-> Đã lọc xong train_verified.txt. Số ảnh lỗi/hỏng: 0
--- Đang kiểm tra sâu: /kaggle/input/face-detection-dataset/images/val ---


  0%|          | 0/3347 [00:00<?, ?it/s]

-> Đã lọc xong val_verified.txt. Số ảnh lỗi/hỏng: 0
✅ Đã tạo eval_300.txt với 300 ảnh


In [14]:
from ultralytics import YOLO

# Xóa cache cũ để tránh xung đột (quan trọng)
import shutil
if os.path.exists('/kaggle/working/runs'):
    shutil.rmtree('/kaggle/working/runs')

# Load model
model = YOLO('yolov8n.pt') 

# Train
model.train(
    data='/kaggle/working/data_final.yaml', # File list đã verify ở bước 1
    epochs=10,
    imgsz=640,
    batch=16,
    
    # --- CÁC THAM SỐ QUAN TRỌNG ĐỂ FIX LỖI ---
    workers=0,      # <--- QUAN TRỌNG NHẤT: Tắt đa luồng load ảnh
    amp=False,      # Tắt mixed precision để tránh lỗi GPU Kaggle
    cache=False,    # Tắt cache RAM để tránh tràn bộ nhớ
    
    project='/kaggle/working/runs/detect',
    name='face_detect_success'
)

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_final.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=face_detect_success, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x77fa11bb9ac0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [15]:
model.val(
    data='/kaggle/working/data_final.yaml',
    split='test',   # 👈 DÙNG TEST, KHÔNG PHẢI VAL
    imgsz=640,
    batch=16,
    workers=0
)


Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.4 ms, read: 517.6±399.3 MB/s, size: 274.6 KB)
val: Scanning /kaggle/input/face-detection-dataset/labels/train... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 680.6it/s 0.4s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/face-detection-dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 4.0it/s 4.7s0.2s
                   all        300       1049      0.894      0.788      0.886       0.57
Speed: 0.3ms preprocess, 2.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /kaggle/working/runs/detect/val


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x77fa0b9c7020>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

precision = 0.894
recall    = 0.788
mAP50     = 0.886
mAP50-95  = 0.570